# Test Stage 4 — Is the averaging correct?

**What this stage checks:** the decimator. That it produces samples at the right rate, that
it scales them correctly, and — the point of the whole block — that it actually reduces noise.

Getting the rate right proves the counting works. Getting the scaling right proves the output
is being read from the correct place inside a 72-bit register. Getting the noise reduction
proves the averaging is genuinely happening rather than samples simply being thrown away.

All three can be wrong independently, so all three are checked.

---

## What you need

| | |
|---|---|
| Stages 2 and 3 | Passing |
| Signal generator | Sine into **IN1** for Checks 1–3. Checks 4–5 want it **switched off or disconnected** |

## The prediction we are testing

For an input sine of amplitude **A** (in ADC counts, peak), with the reference tone sitting
exactly on it, the decimated output should have magnitude

$$\sqrt{I^2 + Q^2} \;=\; A \times 65536$$

That number is built from two known factors:

- the **mixer** contributes ×256 — it multiplies by a reference of peak 32767, halves the
  amplitude in taking the difference term, then shifts right by 6 bits: 32767 ÷ 2 ÷ 64 ≈ 256
- the **decimator** contributes another ×256 — it has unity gain overall, but keeps 8 extra
  bits below the input's least significant bit, and 2⁸ = 256

256 × 256 = 65536. If the measured magnitude is a **power of two** away from this, the output
window inside the decimator is in the wrong place, which is exactly the kind of fault that
produces data that looks reasonable and is wrong by a factor of 2.

---

### Simulation or the real trap?

**This stage never touches the emulated coil.** It measures the acquisition chain itself —
the part that listens — using a bench signal generator plugged into IN1. That is already a
real hardware measurement, so nothing here changes when the trap is connected.

One thing *is* worth doing again for real work. These tests run at a convenient
`F_SIG = 10 MHz` and one fixed averaging setting, and droop, noise and timing all depend on
both. **Before taking trap data, re-run this stage at the frequency and averaging you will
actually use** — around 40 MHz for the coil — so the instrument is characterised where it
is used rather than where it was convenient to test.


## Setup

In [ ]:
import os, mmap, struct, time
import numpy as np

FS_CLK = 125e6          # FPGA processing clock [Hz]

# =============================================================================
#  Acquisition system driver  -  register block at 0x40600000 (sys[6])
# =============================================================================
#  Full register map is in the header of red_pitaya_acq.v and in
#  acquisition_custom/README.md.
#
#  CRITICAL RULE: every access is a SINGLE 32-bit read or write. A bulk mmap
#  slice becomes a burst transfer that the GP-AXI to system-bus bridge cannot
#  acknowledge; it locks the bus and reboots the board. Do not "optimise" the
#  loops below into slice operations.
# =============================================================================
ACQ_BASE = 0x40600000

R_CTRL       = 0x00   # [0] enable  [1] soft reset  [2] test mode  [3] scope tap
R_LO_FINC    = 0x04   # local-oscillator frequency word
R_DEC_LOG2   = 0x08   # log2 of the decimation factor, 6..12
R_SRC_SEL    = 0x0C   # 0 = IN1, 1 = IN2
R_TAP_SEL    = 0x10   # 0 = mixer output, 1 = CIC output
R_TAP_SHIFT  = 0x14   # which 14-bit window of the tapped value to show
R_PEEK_I     = 0x18   # most recent I sample (read-only)
R_PEEK_Q     = 0x1C   # most recent Q sample (read-only)
R_SAMPLE_CNT = 0x20   # output samples since reset (read-only)
R_STATUS     = 0x24   # [0] enabled [1] test mode [2] tap active (read-only)


def s32(v):
    """Reinterpret a 32-bit register value as a signed number.

    Registers come back as unsigned integers, but I and Q are signed. Anything
    with the top bit set is negative, and the value is 2**32 lower than it looks.
    """
    return v - (1 << 32) if v & 0x80000000 else v


class Acq(object):
    def __init__(self, base=ACQ_BASE):
        self.fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        self.m  = mmap.mmap(self.fd, 0x1000, offset=base)

    def w(self, off, val):
        self.m[off:off+4] = struct.pack('<I', val & 0xFFFFFFFF)

    def r(self, off):
        return struct.unpack('<I', self.m[off:off+4])[0]

    # ---- frequency word -----------------------------------------------------
    @staticmethod
    def finc(f_lo):
        """Convert a wanted local-oscillator frequency [Hz] into the register value.

        The oscillator is a counter that represents an angle; its full range is
        one complete turn. Adding `finc` every clock tick advances the angle, so

            f_out = f_clk * finc / 2**32     =>     finc = f_out / f_clk * 2**32

        Step size is 125e6 / 2**32 = 0.0291 Hz.
        """
        return int(round(f_lo / FS_CLK * (1 << 32))) & 0xFFFFFFFF

    @staticmethod
    def finc_to_hz(finc):
        """Inverse of finc(): the frequency the hardware will actually produce."""
        return finc * FS_CLK / (1 << 32)

    # ---- configuration ------------------------------------------------------
    def setup(self, f_lo, dec_log2, src=0, test=False,
              tap_en=False, tap_sel=0, tap_shift=10):
        """Configure and start the chain.

        THE ORDER MATTERS. The decimator holds a running total that assumes one
        fixed decimation factor; changing it while data is flowing leaves a
        meaningless partial sum in the pipeline and produces a burst of nonsense.
        So the sequence is always: disable, write settings, reset, enable.
        """
        self.w(R_CTRL, 0)                       # 1. disable
        self.w(R_LO_FINC,   self.finc(f_lo))    # 2. settings
        self.w(R_DEC_LOG2,  dec_log2)
        self.w(R_SRC_SEL,   src)
        self.w(R_TAP_SEL,   tap_sel)
        self.w(R_TAP_SHIFT, tap_shift)
        ctrl = 1 | (4 if test else 0) | (8 if tap_en else 0)
        self.w(R_CTRL, ctrl | 2)                # 3. enable + one-shot reset
        self.w(R_CTRL, ctrl)                    # 4. running

    def disable(self):
        self.w(R_CTRL, 0)

    def reset(self):
        """Pulse the soft reset, leaving the other control bits as they are."""
        ctrl = self.r(R_CTRL) & 0xD             # keep enable / test / tap bits
        self.w(R_CTRL, ctrl | 2)
        self.w(R_CTRL, ctrl)

    # ---- readback -----------------------------------------------------------
    def peek(self):
        """Most recent (I, Q) output sample, as signed integers."""
        return s32(self.r(R_PEEK_I)), s32(self.r(R_PEEK_Q))

    def count(self):
        return self.r(R_SAMPLE_CNT)


acq = Acq()
print('acquisition system mapped at 0x%08X' % ACQ_BASE)
print('STATUS = 0x%08X' % acq.r(R_STATUS))

In [ ]:
# =============================================================================
#  Internal scope driver  -  register block at 0x40100000 (sys[1])
# =============================================================================
#  Taken unchanged from jupyter_notebooks/phaseshift/scope_phase_check.ipynb, which is
#  already proven on this board.
#
#  NOTE: the equalisation filter this once configured has been REMOVED from
#  the design (it caused 1173 timing violations and was switched off anyway -
#  see CHANGELOG.md). The coefficient registers still exist and still accept
#  writes, they simply have no effect. For these measurements that is an
#  improvement: the scope is now a flat, unmodified path.
# =============================================================================
SC_BASE = 0x40100000
SC_MAP  = 0x40000
BUF_A   = 0x10000
BUF_B   = 0x20000
NMAX    = 1 << 14        # 16384 samples per channel


class Scope(object):
    def __init__(self):
        self.fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        self.m  = mmap.mmap(self.fd, SC_MAP, offset=SC_BASE)

    def w(self, off, val):
        self.m[off:off+4] = struct.pack('<I', val & 0xFFFFFFFF)

    def r(self, off):
        return struct.unpack('<I', self.m[off:off+4])[0]

    def _buf(self, off, n):
        # One 32-bit word at a time - see the warning in the acquisition driver.
        m  = self.m
        ba = bytearray(4 * n)
        for i in range(n):
            o = off + 4 * i
            ba[4*i:4*i+4] = m[o:o+4]
        v = np.frombuffer(bytes(ba), dtype='<u4').astype(np.int64) & 0x3FFF
        # The buffer holds 14-bit two's-complement values. Anything with bit 13
        # set is negative and must have 0x4000 subtracted.
        return np.where(v & 0x2000, v - 0x4000, v).astype(float)

    def acquire(self, dec=1, n=NMAX):
        self.w(0x14, dec)       # scope's own decimation (1 = every sample)
        self.w(0x10, n - 1)     # how many samples to record
        self.w(0x00, 2)         # reset the acquisition engine
        self.w(0x00, 1)         # arm
        self.w(0x04, 1)         # trigger now, from software
        t0 = time.time()
        while self.r(0x04) != 0:            # clears itself when the capture ends
            if time.time() - t0 > 1.0:
                break
        time.sleep(0.005)
        return self._buf(BUF_A, n), self._buf(BUF_B, n), FS_CLK / dec


sc = Scope()
print('internal scope mapped at 0x%08X' % SC_BASE)

---
## Check 1 — Is the output rate exactly 125 MHz ÷ R?

The same measurement as Stage 2, repeated here because it is the foundation everything else
rests on: if the rate is wrong, the scaling and noise numbers cannot be interpreted.

**Passes if:** each measured rate is within about 1% of the prediction. The error is Python's
timing, not the hardware's.

In [ ]:
def measure_rate(dec_log2, seconds=0.30):
    R = 1 << dec_log2
    acq.setup(f_lo=1e6, dec_log2=dec_log2)
    time.sleep(0.02)
    c0 = acq.count(); t0 = time.time()
    time.sleep(seconds)
    c1 = acq.count(); t1 = time.time()
    return R, ((c1 - c0) & 0xFFFFFFFF) / (t1 - t0)


print('%6s %14s %14s %9s  %s' % ('R', 'expected [Hz]', 'measured [Hz]', 'error', 'result'))
print('-' * 60)
ok_all = True
for L in range(6, 13):
    R, rate = measure_rate(L)
    exp = FS_CLK / R
    err = (rate - exp) / exp
    ok  = abs(err) < 0.02
    ok_all &= ok
    print('%6d %14.1f %14.1f %8.2f%%  %s' % (R, exp, rate, 100*err, 'OK' if ok else 'FAIL'))

print()
print('CHECK 1:', 'PASS' if ok_all else 'FAIL')

---
## Check 2 — Measure the input amplitude

We need to know **A** before we can test the prediction. Read it straight off the input with
the tap off, so the scope sees IN1 as usual.

Aim for 2000–6000 counts: comfortably above the noise, comfortably below the 8191 clipping point.

In [ ]:
# ===========================================================================
#  THE ONE LINE YOU MUST EDIT
#
#  F_SIG is what your signal generator is set to. The measurement window is only
#  +-24 kHz wide out of a 62 MHz input range, so a wrong value here does not
#  make the signal weak - it makes it VANISH, exactly as if the cable were
#  unplugged. Check it before trusting anything downstream.
#
#  If it is wrong, the test now says so and sweeps the whole range to tell you
#  the right value. It does not guess silently.
# ===========================================================================
F_SIG = 10.0e6         # <-- YOUR SIGNAL GENERATOR FREQUENCY [Hz]

acq.disable()
time.sleep(0.02)
a, b, fs = sc.acquire(dec=1)

# Amplitude from the root-mean-square rather than the peak: far less sensitive to
# noise and to a stray outlying sample. For a sine, peak = rms * sqrt(2).
A_in  = np.sqrt(2) * np.std(a)

def est_freq(x, fs):
    x = x - x.mean()
    X = np.abs(np.fft.rfft(x * np.hanning(len(x))))
    k = int(np.argmax(X[1:])) + 1
    if 1 <= k < len(X) - 1:
        l, c, r = (np.log(X[k-1]+1e-9), np.log(X[k]+1e-9), np.log(X[k+1]+1e-9))
        k = k + 0.5*(l-r)/(l-2*c+r)
    return k * fs / len(x)

f_in = est_freq(a, fs)

print('input frequency : %.1f Hz' % f_in)
print('input amplitude : %.1f counts (peak, from rms)' % A_in)
print('peak sample     : %.0f counts (clipping starts at 8191)' % np.abs(a).max())
print()
print('predicted decimated magnitude = A x 65536 = %.3e' % (A_in * 65536))

if abs(f_in - F_SIG) / F_SIG > 0.01:
    print('\n** F_SIG updated to the measured %.1f Hz **' % f_in)
    F_SIG = f_in

---
## Check 3 — Is the scaling right?

Put the reference tone exactly on the signal, so the decimated output is a steady complex
value, and read it directly out of the peek registers. Repeat for every decimation factor —
the magnitude should be **the same for all of them**, because the decimator has unity gain by
design. Only the noise on it should change.

**Passes if:** the measured magnitude is within about 15% of A × 65536, at every R.

**If it fails:** a ratio that is a power of two (½, 2, 4…) means the output window inside the
decimator sits in the wrong place. A ratio that drifts with R means the window is not moving
with the decimation factor as it should.

In [ ]:
def peek_mag(n=64):
    """Average magnitude over several peek reads, to smooth out noise."""
    vals = []
    for _ in range(n):
        i, q = acq.peek()
        vals.append(np.hypot(float(i), float(q)))
        time.sleep(0.001)
    return float(np.mean(vals)), float(np.std(vals))


predicted = A_in * 65536

print('%6s %16s %16s %10s  %s' % ('R', 'predicted', 'measured', 'ratio', 'result'))
print('-' * 66)
ok_all = True
results = {}
for L in range(6, 13):
    acq.setup(f_lo=F_SIG, dec_log2=L)
    time.sleep(0.10)                    # let the filter fill before reading
    mag, sd = peek_mag()
    results[L] = (mag, sd)
    ratio = mag / predicted
    ok = 0.85 < ratio < 1.15
    ok_all &= ok
    print('%6d %16.3e %16.3e %10.3f  %s'
          % (1 << L, predicted, mag, ratio, 'OK' if ok else 'FAIL'))

print()
print('CHECK 3:', 'PASS' if ok_all else 'FAIL')
if not ok_all:
    print('   a ratio near a power of two points at the output bit window;')
    print('   see SECTION 4 of red_pitaya_acq_cic.v')

---
## Check 4 — Does averaging actually reduce noise?

**This is the reason the block exists**, so it deserves a direct test.

**Turn the signal generator off, or unplug it from IN1.** We want to measure noise, and a
large signal sitting on top makes that harder than it needs to be.

Averaging R samples should improve the signal-to-noise ratio by roughly √R — averaging 1024
samples should reduce the noise by about 32 times. In practice a four-stage filter does
slightly *better* than √R, because it is sharper than a plain block average and so lets
through a little less noise. The software model predicts about 1.46× better, so expect
somewhere around 1.4–1.5 times the √R figure.

**Passes if:** the noise falls by at least √R each time, and the trend follows the √R line.

**If it fails:** noise that does not fall at all means the averaging is not happening —
samples are being discarded without being summed, which would look completely normal on
every other check in this notebook.

In [ ]:
print('*** Switch the signal generator OFF, and ideally terminate IN1 with 50 ohm ***')
print()

def peek_noise(n=400):
    """Standard deviation of I over many peek reads, in ADC-count units.

    The decimator has unity gain but carries 8 extra fractional bits, so dividing
    by 256 puts the answer back into ADC counts and makes the numbers comparable
    across decimation factors.
    """
    vals = []
    for _ in range(n):
        i, _q = acq.peek()
        vals.append(float(i))
        time.sleep(0.0005)
    return float(np.std(vals)) / 256.0


Rs, sd = [], []
print('%6s %16s' % ('R', 'noise [counts]'))
print('-' * 24)
for L in range(6, 13):
    acq.setup(f_lo=F_SIG, dec_log2=L)
    time.sleep(0.15)
    nz = peek_noise()
    Rs.append(1 << L); sd.append(nz)
    print('%6d %16.4f' % (1 << L, nz))

Rs = np.array(Rs, float); sd = np.array(sd)

# -------------------------------------------------------------------------
#  Separate the two things that make up the measured noise.
#
#  The decimator reduces noise by NARROWING THE BANDWIDTH. It removes noise
#  lying OUTSIDE the final band; anything already INSIDE that band is passed
#  through untouched. So the measured spread is two components added in
#  quadrature:
#
#      variance  =  C^2   +   K / R
#                   ^^^^       ^^^^^
#                   floor      broadband noise, falls as 1/R in power
#
#  C is whatever sits inside the band - a discrete interferer near the
#  reference tone, or low-frequency drift. No amount of averaging removes it.
#
#  Demanding the RAW figure follow sqrt(R) all the way to R=4096 assumes a
#  perfect noise-free environment, which no real analogue front end has. The
#  honest test is whether the BROADBAND part follows sqrt(R) - that is the part
#  the decimator is responsible for.
# -------------------------------------------------------------------------
# Fit  variance = C2 + K*x  with x = 1/R, i.e. a straight line.
#
# np.polyfit rather than np.linalg.lstsq: the numpy on the Red Pitaya is old and
# rejects rcond=None with "TypeError: a float is required". The same warning is
# already written into jupyter_notebooks/phaseshift/scope_phase_check.ipynb - this notebook
# should have heeded it.
#
# polyfit returns the highest power first, so for a straight line that is
# [slope, intercept] = [K, C2].
K, C2 = np.polyfit(1.0 / Rs, sd ** 2, 1)
C = np.sqrt(max(C2, 0.0))

print()
print('FIT  std(R) = sqrt(C^2 + K/R)')
print('  floor C            : %8.2f counts  (%.1f dBFS) - averaging cannot remove this'
      % (C, 20 * np.log10(max(C, 1e-9) / 8191)))
print('  broadband at R=64  : %8.2f counts' % np.sqrt(K / 64))
print()

white = np.sqrt(K / Rs)
print('%6s %12s %12s %12s %8s  %s'
      % ('R', 'measured', 'model', 'broadband', 'vs sqrt(R)', 'result'))
print('-' * 74)
ok_all = True
for r, m in zip(Rs, sd):
    model = np.sqrt(C2 + K / r)
    w     = np.sqrt(K / r)
    ratio = (np.sqrt(K / 64) / w) / np.sqrt(r / 64.0)     # 1.00 = ideal
    ok    = 0.85 < ratio < 1.20
    ok_all &= ok
    print('%6d %12.2f %12.2f %12.2f %8.3f  %s'
          % (r, m, model, w, ratio, 'OK' if ok else 'LOW'))

resid = np.sqrt(np.mean((sd - np.sqrt(C2 + K / Rs)) ** 2))
print()
print('fit residual : %.2f counts   (a good fit means the two-component picture is right)' % resid)
print()
print('CHECK 4:', 'PASS' if (ok_all and resid < 0.1 * sd.mean()) else 'FAIL')
print()
if C > 5:
    print('NOTE: the %.0f-count floor is NOT a fault in the decimator - the broadband' % C)
    print('part above follows sqrt(R) exactly. It is something arriving within the')
    print('measurement band, which is only a few kHz wide around your reference tone.')
    print('Run the diagnostic cell below to find out whether it is a discrete')
    print('interferer near that frequency or something in the chain itself.')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7.5, 5))
plt.loglog(Rs, sd, 'o', ms=8, label='measured')
plt.loglog(Rs, np.sqrt(C2 + K / Rs), '-', label=r'fit: $\sqrt{C^2 + K/R}$')
plt.loglog(Rs, np.sqrt(K / Rs), '--', label='broadband part alone')
plt.axhline(C, color='r', ls=':', label='floor C = %.0f counts' % C)
plt.xlabel('decimation factor R')
plt.ylabel('noise on the output [ADC counts]')
plt.title('The broadband part follows $1/\\sqrt{R}$ exactly.\n'
          'The flattening is a floor inside the measurement band, not a filter fault.')
plt.grid(True, which='both', alpha=0.4)
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ---------------------------------------------------------------------------
#  Diagnostic: is the floor a discrete interferer, or the chain itself?
#
#  The whole acquisition chain is a BANDPASS filter centred on the reference
#  tone: anything more than a few kHz away is mixed off and then removed. So a
#  floor that survives averaging must be arriving close to the reference
#  frequency.
#
#  Move the reference and re-measure:
#     floor DROPS when you move away from F_SIG -> a discrete interferer sits
#         near F_SIG. Likely the generator still emitting, or pickup on an
#         unterminated input.
#     floor STAYS the same wherever you look -> broadband noise inside the
#         chain, independent of frequency.
# ---------------------------------------------------------------------------
print('Signal generator should still be OFF for this.')
print()

test_freqs = [F_SIG, 2e6, 5e6, 12e6, 25e6]

print('%14s %16s' % ('reference [Hz]', 'floor [counts]'))
print('-' * 32)
floors = []
for f in test_freqs:
    # Measure at two decimations and solve for the floor, which is quicker than
    # a full sweep and enough to separate floor from broadband part.
    vals = []
    for L in (6, 12):
        acq.setup(f_lo=f, dec_log2=L)
        time.sleep(0.15)
        vals.append(peek_noise(200))
    s64, s4096 = vals
    # variance = C^2 + K/R, two equations two unknowns
    Kf  = (s64**2 - s4096**2) / (1.0/64 - 1.0/4096)
    Cf2 = s4096**2 - Kf / 4096
    Cf  = np.sqrt(max(Cf2, 0.0))
    floors.append(Cf)
    tag = '   <-- your signal frequency' if abs(f - F_SIG) < 1 else ''
    print('%14.0f %16.2f%s' % (f, Cf, tag))

floors = np.array(floors)
print()
if floors[0] > 2 * np.median(floors[1:]):
    print('VERDICT: the floor is much higher at your signal frequency than elsewhere.')
    print('  -> a discrete interferer sits near %.4f MHz.' % (F_SIG/1e6))
    print('  -> most likely the generator is still emitting with its output off,')
    print('     or IN1 is unterminated and picking it up through the air.')
    print('  -> try: disconnect at the IN1 end, and terminate IN1 with 50 ohm.')
else:
    print('VERDICT: the floor is similar at every reference frequency.')
    print('  -> it is not a single interferer; it is broadband, in the analogue')
    print('     front end or the board environment.')
    print('  -> this sets the ultimate sensitivity of the instrument, and is')
    print('     worth recording as such rather than treated as a fault.')

plt.figure(figsize=(7, 4))
plt.semilogx(np.array(test_freqs)/1e6, floors, 'o-')
plt.xlabel('reference tone frequency [MHz]'); plt.ylabel('floor [ADC counts]')
plt.title('Does the un-averageable floor depend on where you look?')
plt.grid(True, which='both', alpha=0.4); plt.tight_layout(); plt.show()

---
## Check 5 — Look at the decimated output directly

A last, more visual check. Tap the **decimator** output rather than the mixer (`TAP_SEL = 1`)
and capture it with the scope.

The scope samples every clock tick, but the decimator only produces a new value every R ticks.
So the trace should look like a **staircase**, each step exactly R samples wide. Measuring the
step width is an independent confirmation of the decimation factor — one that does not depend
on Python's timing at all, unlike Check 1.

Use a small R here. At R = 1024 a full 16384-sample capture would contain only 16 steps.

In [ ]:
print('*** Turn the signal generator back ON for this check ***')
print()

L_VIEW = 8                       # R = 256, giving 64 steps in one capture
R_VIEW = 1 << L_VIEW

acq.setup(f_lo=F_SIG - 2e3, dec_log2=L_VIEW,   # small offset so there is something to see
          tap_en=True, tap_sel=1, tap_shift=14)
time.sleep(0.10)

# Auto-range the tap window as in the earlier notebooks.
best = 14
for shift in range(18, -1, -1):
    acq.w(R_TAP_SHIFT, shift)
    time.sleep(0.01)
    a, b, _ = sc.acquire(dec=1, n=4096)
    if max(np.abs(a).max(), np.abs(b).max()) > 7500:
        break
    best = shift
    if max(np.abs(a).max(), np.abs(b).max()) > 2500:
        break
acq.w(R_TAP_SHIFT, best)

a, b, fs = sc.acquire(dec=1)

# Find where the value changes, and measure the gaps between those points.
edges = np.where(np.diff(a) != 0)[0]
steps = np.diff(edges) if len(edges) > 2 else np.array([0])

print('TAP_SHIFT               : %d' % best)
print('decimation factor set   : %d' % R_VIEW)
print('measured step width     : %.1f samples (median)' % np.median(steps))
print('number of steps in view : %d' % len(edges))
print()

ok = abs(np.median(steps) - R_VIEW) < 2
print('CHECK 5:', 'PASS' if ok else 'FAIL - step width does not match R')

n_show = min(len(a), 12 * R_VIEW)
plt.figure(figsize=(10, 4))
plt.step(np.arange(n_show) / fs * 1e6, a[:n_show], where='post', label='I')
plt.step(np.arange(n_show) / fs * 1e6, b[:n_show], where='post', label='Q')
plt.xlabel('time [us]'); plt.ylabel('tapped value')
plt.title('Decimator output: each step should be exactly R = %d clock ticks wide' % R_VIEW)
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

---
## Stage 4 summary

| Check | What it proves |
|---|---|
| 1 — output rate | The counting is right: one sample out per R samples in |
| 2 — input amplitude | Gives us the number the prediction is built from |
| 3 — scaling | The output is read from the right place inside a 72-bit register |
| 4 — noise | The averaging is genuinely happening, not just discarding samples |
| 5 — staircase | Independent confirmation of the decimation factor, with no reliance on software timing |

**All five passing** means the front three-quarters of the acquisition chain is sound: the
reference tone is accurate, the frequency shift lands where it should and the right way round,
and the averaging works and delivers the noise reduction it is there for.

What is still untested is the **droop** described in the README — the decimator is quieter for
signals near the edge of the measurement window than at its centre. That is the correction
filter's job, and Test Stage 5 once it exists.

**Before finishing**, leave the hardware in a clean state:

In [ ]:
acq.disable()
print('acquisition disabled, scope restored to its normal inputs')
print('STATUS = 0x%08X' % acq.r(R_STATUS))